In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import pickle


Load the dataset

In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Preprocessing the data
- [x] Gender
- [x] Geography


#### Data dropping

In [3]:
data = data.drop(['RowNumber','CustomerId','Surname'], axis = 1) #Dropping irrelevent data
data.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


#### Encoding categorical variable

##### Label encoder

In [4]:
Label_encoder_gender = LabelEncoder()
data['Gender'] = Label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


##### One hot encoding

In [5]:
ohe_geo = OneHotEncoder()
geo_encoder = ohe_geo.fit_transform(data[['Geography']])

In [6]:
ohe_geo.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [7]:
geo_encoded = pd.DataFrame(geo_encoder.toarray(), columns=ohe_geo.get_feature_names_out())

In [8]:
data = pd.concat([data.drop('Geography', axis = 1), geo_encoded], axis = 1)


In [9]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


#### Saving the encoders and scaler

In [10]:
with open('Label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(Label_encoder_gender,file)
with open('ohe_geo.pkl', 'wb') as file:
    pickle.dump(ohe_geo, file)

#### Divide dataset into dependant and independant feautures

In [11]:
X = data.drop('Exited', axis=1) #Independant
Y = data['Exited'] #Dependatn

### Splitting the data in training and testing sets

In [12]:
X_Train, X_Test, Y_Train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=67)

### Scaling the data

In [13]:
scaler = StandardScaler()
X_Train = scaler.fit_transform(X_Train)
X_Test = scaler.transform(X_Test)

In [14]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

# ANN Implementation

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

## Building model

In [16]:
model = Sequential([

    Dense(64, activation='relu', input_shape=(X_Train.shape[1], )), #Hidden Layer 1, connected with input layer
    Dense(32, activation='relu'), #Hidden Layer 2
    Dense(1, activation='sigmoid') #Output layer

])

In [17]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## Model compile

In [18]:
opt = tf.keras.optimizers.Adam(learning_rate=0.025) #Variable learning rate that we can set
loss = tf.keras.losses.BinaryCrossentropy() #Setting loss function

In [19]:
model.compile(optimizer=opt, loss=loss, metrics=['accuracy']) 


You may also give direct arguments in the complie functions
```python
model.compile(optimizer="adam", loss="binary_crossentroy", metrics=['accuracy'])
```
But this way you cannot set your custom learning rate

### Setup TensorBoard

In [20]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [21]:
tf_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

### Setup early stopping

In [22]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

## Training the model

In [23]:
history = model.fit(

    X_Train, Y_Train, validation_data=(X_Test, Y_test), epochs=100,
    callbacks = [tf_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 3s 4ms/step - loss: 0.4056 - accuracy: 0.8281 - val_loss: 0.3496 - val_accuracy: 0.8610
Epoch 2/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3603 - accuracy: 0.8506 - val_loss: 0.3542 - val_accuracy: 0.8580
Epoch 3/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3600 - accuracy: 0.8522 - val_loss: 0.3560 - val_accuracy: 0.8610
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3560 - accuracy: 0.8515 - val_loss: 0.3499 - val_accuracy: 0.8635
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3504 - accuracy: 0.8539 - val_loss: 0.3621 - val_accuracy: 0.8570
Epoch 6/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3470 - accuracy: 0.8540 - val_loss: 0.3499 - val_accuracy: 0.8610
Epoch 7/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3467 - accuracy: 0.8589 - val_loss: 0.3486 - val_accuracy: 0.86

In [24]:
model.save('model.h5') #Saving the model as keras
model = tf.keras.models.load_model('model.h5')

c:\Users\goggl\ANN Example\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


### Load tensor board

In [25]:
%reload_ext tensorboard

In [26]:
import pkg_resources
%tensorboard --logdir logs/fit/

ModuleNotFoundError: No module named 'pkg_resources'